# 06 – Export for Power BI

Reshapes the outputs of notebooks 01-05 into a small **star schema** for Power BI: dimension tables (dates, holdings, pies, series) and fact tables in long format (one row per date and series, per pie and measure, and so on). Long tables are easy to filter and slice in Power BI; wide tables are not.

All files are written as Parquet to `output/powerbi/` (not tracked by Git). Parquet keeps the data types, so there is no confusion between a decimal point and a decimal comma in a Dutch Power BI installation.

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from portfolio_analysis.config import BLOCKS, ECON_COLUMNS

In [2]:
PROCESSED_DIR = Path("../data/processed")
REFERENCE_DIR = Path("../data/reference")
EXPORT_DIR = Path("../output/powerbi")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

PIE_BENCHMARK_LABELS = {
    "Kwaliteit": "IWQU (MSCI World Quality)",
    "Thematisch": "IWDA (MSCI World)",
    "AI Boom": "AIQU (AI & Big Data)",
    "Energie": "70% WNRG / 30% NUCL",
    "Mijnbouw": "65% GDX / 35% COPX",
    "Multipolaire Wereldorde": "EIMI (MSCI EM IMI)",
}
ECON_LABELS = {"econ_usd": "USD", "econ_eur": "EUR", "econ_gbp": "GBP", "econ_chf": "CHF", "econ_cad": "CAD", "econ_other_dm": "Other DM", "econ_em": "EM"}

exports = {}

## Dimensions: pies and holdings

In [3]:
holdings = pd.read_parquet(PROCESSED_DIR / "holdings.parquet")
symbols = pd.read_csv(REFERENCE_DIR / "yahoo_symbols.csv", keep_default_na=False, na_values=[""])
block_of = {pie: block for block, pies in BLOCKS.items() for pie in pies}

dim_pie = (
    holdings.groupby("pie", as_index=False)
    .agg(target_weight=("weight_in_portfolio", "sum"), n_positions=("ticker", "count"))
    .assign(block=lambda d: d["pie"].map(block_of), benchmark=lambda d: d["pie"].map(PIE_BENCHMARK_LABELS))
)
pie_order = [pie for pies in BLOCKS.values() for pie in pies]
dim_pie["sort_order"] = dim_pie["pie"].map({pie: i for i, pie in enumerate(pie_order)})

dim_holding = holdings[[
    "ticker", "name", "isin", "pie", "asset_type", "category", "country", "exchange",
    "listing_currency", "main_currency", "weight_in_portfolio", "weight_in_pie", "dividend_yield", "ter",
]].merge(symbols[["ticker", "yahoo_symbol", "series_type", "proxy_symbol"]], on="ticker", how="left", validate="one_to_one")
dim_holding["block"] = dim_holding["pie"].map(block_of)
dim_holding["has_proxy"] = dim_holding["proxy_symbol"].notna()

holding_currency = holdings.melt(id_vars="ticker", value_vars=ECON_COLUMNS, var_name="currency", value_name="share")
holding_currency["currency"] = holding_currency["currency"].map(ECON_LABELS)

assert abs(dim_pie["target_weight"].sum() - 1) < 1e-9
assert len(dim_holding) == 55
exports.update({"dim_pie": dim_pie, "dim_holding": dim_holding, "holding_currency": holding_currency})
dim_pie

,pie,target_weight,n_positions,block,benchmark,sort_order
0,AI Boom,0.10,8,A,AIQU (AI & Big Data),2
1,Energie,0.18,10,B,70% WNRG / 30% NUCL,3
2,Kwaliteit,0.27,17,A,IWQU (MSCI World Quality),0
3,Mijnbouw,0.17,9,B,65% GDX / 35% COPX,4
4,Multipolaire Wereldorde,0.15,6,B,EIMI (MSCI EM IMI),5
5,Thematisch,0.13,5,A,IWDA (MSCI World),1


## Facts: daily returns, growth and drawdown per series

One row per date and series. Growth (value of 1 EUR) and drawdown are calculated here rather than in DAX: a running product is simple in Python and slow and complex in DAX.

In [4]:
backtest = pd.read_parquet(PROCESSED_DIR / "backtest_returns.parquet")
pie_daily = pd.read_parquet(PROCESSED_DIR / "pie_returns.parquet")
pie_weights = dim_pie.set_index("pie")["target_weight"]

blocks = pd.DataFrame({
    f"Block {block}": pie_daily[pies].mul(pie_weights[pies]).sum(axis=1, min_count=len(pies)) / pie_weights[pies].sum()
    for block, pies in BLOCKS.items()
})
wide = pd.concat([backtest, blocks, pie_daily], axis=1, sort=True)
wide.index.name = "date"

series_group = {name: "Portfolio" for name in backtest.columns if name.startswith("Portfolio")}
series_group.update({name: "Benchmark" for name in backtest.columns if not name.startswith("Portfolio")})
series_group.update({name: "Block" for name in blocks.columns})
series_group.update({name: "Pie" for name in pie_daily.columns})

frames = []
for name in wide.columns:
    r = wide[name].dropna()
    growth = (1 + r).cumprod()
    frames.append(pd.DataFrame({
        "date": r.index, "series": name, "return": r.to_numpy(),
        "growth": growth.to_numpy(), "drawdown": (growth / growth.cummax() - 1).to_numpy(),
    }))
fact_returns = pd.concat(frames, ignore_index=True)

dim_series = pd.DataFrame({"series": list(series_group), "group": list(series_group.values())})
exports.update({"fact_returns": fact_returns, "dim_series": dim_series})
fact_returns.groupby("series").agg(start=("date", "min"), end=("date", "max"), last_growth=("growth", "last")).round(3)

C:\Users\caswv\AppData\Local\Temp\ipykernel_7008\378638865.py:29: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  fact_returns.groupby("series").agg(start=("date", "min"), end=("date", "max"), last_growth=("growth", "last")).round(3)


,start,end,last_growth
series,,,
AI Boom,2019-01-04,2026-11-03,18.184
Block A,2019-01-04,2026-11-03,5.375
Block B,2019-01-04,2026-11-03,3.767
Energie,2019-01-04,2026-11-03,5.332
Kwaliteit,2019-01-04,2026-11-03,3.657
MSCI World (IWDA),2019-01-03,2026-11-03,2.986
Main benchmark (monthly),2019-01-04,2026-11-03,2.617
Mijnbouw,2019-01-04,2026-11-03,4.785
Multipolaire Wereldorde,2019-01-04,2026-11-03,1.448


In [5]:
dates = pd.date_range(fact_returns["date"].min(), fact_returns["date"].max(), freq="D")
dim_date = pd.DataFrame({"date": dates})
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["month"] = dim_date["date"].dt.month
dim_date["year_month"] = dim_date["date"].dt.strftime("%Y-%m")
dim_date["is_trading_day"] = dim_date["date"].isin(fact_returns["date"])
exports["dim_date"] = dim_date
dim_date.shape

(2862, 6)

## Facts: performance and risk tables in long format

In [6]:
def long_table(df, id_vars, var_name, value_name="value"):
    """Melt a wide table into long format and drop empty values."""
    return df.melt(id_vars=id_vars, var_name=var_name, value_name=value_name).dropna(subset=[value_name])


performance_full = pd.read_parquet(PROCESSED_DIR / "performance_full.parquet")
performance_sub = pd.read_parquet(PROCESSED_DIR / "performance_subperiods.parquet")
performance_pies = pd.read_parquet(PROCESSED_DIR / "performance_pies.parquet")
real_returns = pd.read_parquet(PROCESSED_DIR / "real_returns.parquet")

metric_columns = ["ann_return", "ann_vol", "sharpe", "sortino", "max_drawdown", "beta", "tracking_error", "information_ratio"]
fact_performance = pd.concat([
    long_table(performance_full[["series", *metric_columns]], ["series"], "metric").assign(period="2019-now"),
    long_table(performance_sub[["series", "period", *[c for c in metric_columns if c in performance_sub]]], ["series", "period"], "metric"),
], ignore_index=True)
fact_pie_performance = long_table(performance_pies.drop(columns="start"), ["pie"], "metric")
fact_real_returns = long_table(real_returns, ["series"], "measure")

exports.update({"fact_performance": fact_performance, "fact_pie_performance": fact_pie_performance, "fact_real_returns": fact_real_returns})
fact_performance.head()

,series,metric,value,period
0,Portfolio (monthly rebalanced),ann_return,0.208909,2019-now
1,Portfolio (buy-and-hold),ann_return,0.205942,2019-now
2,Main benchmark (monthly),ann_return,0.127165,2019-now
3,MSCI World (IWDA),ann_return,0.145754,2019-now
4,Portfolio (monthly rebalanced),ann_vol,0.170439,2019-now


In [7]:
risk = {name: pd.read_parquet(PROCESSED_DIR / f"{name}.parquet") for name in [
    "risk_correlation", "risk_rolling", "risk_var", "risk_contribution", "risk_stress",
    "risk_var_backtest", "risk_factor_betas", "risk_scenarios", "risk_taiwan",
]}

exports["fact_correlation"] = risk["risk_correlation"]
exports["fact_rolling"] = risk["risk_rolling"]
exports["fact_var"] = long_table(risk["risk_var"], ["window", "confidence", "method"], "measure")
exports["fact_var_backtest"] = risk["risk_var_backtest"]
exports["fact_risk_contribution"] = long_table(risk["risk_contribution"], ["pie"], "period")
exports["fact_stress"] = long_table(risk["risk_stress"], ["series"], "scenario")
exports["fact_scenarios"] = long_table(risk["risk_scenarios"], ["series"], "scenario")
exports["fact_factor_betas"] = long_table(risk["risk_factor_betas"], ["pie"], "factor")
exports["fact_taiwan"] = long_table(
    risk["risk_taiwan"][["ticker", "pie", "weight_in_portfolio", "Blockade", "Invasion"]],
    ["ticker", "pie", "weight_in_portfolio"], "variant", "shock",
).assign(contribution=lambda d: d["weight_in_portfolio"] * d["shock"])
sorted(exports)

['dim_date',
 'dim_holding',
 'dim_pie',
 'dim_series',
 'fact_correlation',
 'fact_factor_betas',
 'fact_performance',
 'fact_pie_performance',
 'fact_real_returns',
 'fact_returns',
 'fact_risk_contribution',
 'fact_rolling',
 'fact_scenarios',
 'fact_stress',
 'fact_taiwan',
 'fact_var',
 'fact_var_backtest',
 'holding_currency']

## Write

In [8]:
for name, table in exports.items():
    table.to_parquet(EXPORT_DIR / f"{name}.parquet", index=False)

pd.DataFrame({"table": list(exports), "rows": [len(t) for t in exports.values()], "columns": [t.shape[1] for t in exports.values()]})

,table,rows,columns
0,dim_pie,6,6
1,dim_holding,55,19
2,holding_currency,385,3
3,fact_returns,24301,5
4,dim_series,12,2
5,dim_date,2862,6
6,fact_performance,65,4
7,fact_pie_performance,54,3
8,fact_real_returns,15,3
9,fact_correlation,36,3
